<a href="https://colab.research.google.com/github/Jorge-Ruiz-Troccoli/Data-Science-II/blob/main/Clase%2010/Ejemplo_clasificaci%C3%B3n_PARTE_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introducción al ML
### Coderhouse - Data Science
Profe Jorge Ruiz

In [ ]:
import numpy as np
import pandas as pd


In [ ]:
df= pd.read_csv("encuesta_dueños_perros.csv", encoding="utf-8")
df.head()

In [ ]:
df.info()

In [ ]:
df.Sabor_pescado.value_counts()

In [ ]:
df.Sabor_pescado.value_counts(normalize=True)*100

In [ ]:
print(f"total de registros, incluyendo NaN: {df.Peso.size}")
print(f"total de registros, sin incluir los NaN: {df.Peso.count()}")

#size y count no son iguales

In [ ]:
df.groupby(["Tamaño", "Sabor_pescado"]).count()
# conversando con especialistas del tema, el tamaño puede ser una variable clave en la decisión

In [ ]:
df.isnull().sum()

In [ ]:
#proporción de valores faltantes en el conjunto de datos.

df.isnull().sum()/len(df)*100

In [ ]:
df.duplicated().sum()

# no es recomendable eliminar estos datos duplicados, son pocas preguntas que provienen de una encuesta y puede haber muchos casos identicos

In [ ]:
#intentamos cambiar el peso a float y da error

df["Peso"]=df["Peso"].astype(float)

In [ ]:
# Reemplazar comas por puntos y convertir a float
df["Peso"] = df["Peso"].str.replace(",", ".").astype(float)

In [ ]:
df.describe().round(0)

In [ ]:
import seaborn as sns
sns.boxplot(df.Peso)

In [ ]:
#histograma
sns.histplot(data=df, x="Peso", kde=True, color="skyblue")

##Detección de outliers:

In [ ]:
def filtro_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    filtro = (df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR)) | (df[col] < 1) | (df[col] > 80)

    # el método where() se utiliza para seleccionar elementos de un objeto DataFrame o Serie según una condición booleana
    #y reemplazar los elementos que no cumplen la condición con un valor especificado.

    df[col] = df[col].where(~filtro, np.nan)
    return df

# puede aplicarse a todas las columnas numéricas iterando sobre cada una de ellas en un ciclo, por ejemplo un for
df_so = filtro_outliers(df, 'Peso')
df_so


In [ ]:
#veamos por pasos como funciona el where con un ejemplo

df_prueba=pd.DataFrame({"col1":[2,4,-8,5]})
df_prueba



In [ ]:
filtro=(df_prueba.col1 <= 2)

df_prueba[filtro]

In [ ]:
df_prueba["col1"].where(~filtro,"error")


In [ ]:
histograma
sns.histplot(data=df_so, x="Peso", kde=True, color="skyblue")



In [ ]:
# Identificar y eliminar valores imposibles
df.describe().round(0)

In [ ]:
tamaños=df.Tamaño.unique()
tamaños

In [ ]:
df_Grande= df_so[df_so.Tamaño=="Grande"]
df_Mediano= df_so[df_so.Tamaño=="Mediano"]
df_Pequeño =df_so[df_so.Tamaño=="Pequeño"]

In [ ]:
df_Pequeño

In [ ]:
# rellenamos lo valores faltantes con la mediana segun el tamaño del perro

df_Grande["Peso"].fillna(df_Grande["Peso"].median(), inplace=True)
df_Mediano["Peso"].fillna(df_Mediano["Peso"].median(), inplace=True)
df_Pequeño["Peso"].fillna(df_Pequeño["Peso"].median(), inplace=True)

#ahora concatenamos ambos dataframes

df_final= pd.concat([df_Pequeño, df_Mediano,df_Grande], axis=0)
df_final

In [ ]:
df_final.isnull().sum()

In [ ]:
df_final.describe()

# Un mejor camino y el recomendado

In [ ]:
df2=df.groupby(['Tamaño'])['Peso'].median().reset_index()
# Convertir la mediana a enteros
df2

In [ ]:
# Crear un diccionario de mapeo de los valores de df2
mapping_dict = df2.set_index('Tamaño')['Peso'].to_dict()
mapping_dict


In [ ]:
# Mapear los valores de df['Area dedicated'] a los valores correspondientes de df2['Salary(ARS)']
df_final2= df
df_final2['Peso2'] = df_final2['Tamaño'].map(mapping_dict)
df_final2


In [ ]:
# Asignar los valores de otra columna a 'Salary(ARS)' donde corresponda
df_final2['Peso'].fillna(df_final2['Peso2'], inplace=True)
df_final2

In [ ]:
df_final2.describe()

##Ordinal encoding

In [ ]:
#Asignación manual a traves de un diccionario
diccionario = {'Pequeño':1, 'Mediano':2, 'Grande':3}

df_final['Tamaño_OE']= df_final['Tamaño'].map(diccionario)
df_final.head()

#Vamos a estudiar la correlación de spearman

In [ ]:

from scipy.stats import spearmanr

#Ho= no hay correlación entre las variables
#Ha= si hay correlación entre las variables

# Calcular coeficiente de correlación de rangos de Spearman
coeficiente, p_valor = spearmanr(df_final["Peso"], df_final["Tamaño_OE"])

print("coeficiente:", coeficiente)
print("P-valor:", p_valor)

#leer más en https://estadisticacienciassocialesr.rbind.io/relaci%C3%B3n-entre-variables-el-an%C3%A1lisis.html

Recordar: cuando el valor de p es menor que 0.05, generalmente se considera que hay evidencia estadística significativa para rechazar la hipótesis nula.

In [ ]:
df_final.columns

##Hagamos un Decision Tree (árbol de decisión)

In [ ]:
# Split en train y test
X= df_final.drop(columns=['Sabor_pescado', 'Tamaño'])
y= df_final['Sabor_pescado']

X_new=pd.get_dummies(X,drop_first=True)#  la función get_dummies solo se aplica a variables categóricas o de tipo objeto
X_new

In [ ]:

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_new, y, test_size=0.3, random_state=42)

In [ ]:
X_test

¿Qué son los hiperparámetros?

Los hiperparámetros son variables de configuración externa que los científicos de datos utilizan para administrar el entrenamiento de modelos de machine learning. A veces llamados hiperparámetros de modelos, los hiperparámetros se configuran de manera manual antes de entrenar un modelo.

https://aws.amazon.com/es/what-is/hyperparameter-tuning/


Vamos a regularizar el algoritmo. Recuerden que este tiene distintos parametros. En particular, podemos elegir si utiliza Gini o Entropia para calcular la impureza de un splitting. Generalmente no hay diferencia pero por definicion Gini puede favorecer mas la clase mas frecuente. La ventaja es que es mas rapida.

Las opciones que tenemos en sklearn son:

max_depth: por defecto es None, controla la profundidad del arbol.

min_samples_split: establece el minimo numero de muestras que debe tener un nodo para poder seguir partiendolo.

min_samples_leaf: el minimo numero de muestras que debe tener una hoja.

min_impurity_decrease= Si el valor de la impureza disminuida por una división es menor que min_impurity_decrease, entonces la división no se realiza y el nodo se convierte en una hoja final.

max_leaf_nodes: maxima cantidad de hojas.

max_features: maxima cantidad de features evaluados en un splitting.

Si uno sube los valores minimos o baja los maximos, esta restringiendo al arbol y regularizando el modelo.

Existen otros metodos de regularizacion como por ejemplo el pruning o podado de arboles en el que se entrena sin restricciones y luego se eliminan nodos innecesarios

In [ ]:
# Entrenar el arbol
from sklearn import tree
clf = tree.DecisionTreeClassifier(max_depth=4, min_samples_split=25, min_samples_leaf=10)
clf = clf.fit(X_train, y_train)

In [ ]:
y_train.unique()

In [ ]:
# Graficando
from matplotlib import pyplot as plt
fig = plt.figure(figsize=(20,10))
_ = tree.plot_tree(clf,feature_names=X_train.columns,
                   class_names=y_train.unique(),
                   filled=True)

Se va fijando en el peso... Pero nosotros ya sabemos que es lo único que importa. Este es un limitante de los DTs. Hacen cortes en cada parametro por separado, no piensa en combinarlos inteligentemente. Esta es la razón por la que tampoco hace falta estandarizar los datos.

In [ ]:

# En un árbol de decisión, puedes obtener la importancia de las variables utilizando el atributo feature_importances_ del clasificador entrenado.
# nos devuelve la importancia la de las variables en un árbol de decisión y se basa en la medida de impureza utilizada durante la construcción del árbol, en este caso gini.

importances = clf.feature_importances_

# Obtener los nombres de las variables
feature_names = X_train.columns

# Crear un DataFrame con las importancias de las variables
importance_df = pd.DataFrame({'Variable': feature_names, 'Importance': importances})

# Ordenar el DataFrame por importancia descendente
importance_df = importance_df.sort_values('Importance', ascending=False)


importance_df

 # se recomienda revisar https://scikit-learn.org/stable/auto_examples/ensemble/plot_forest_importances.html

In [ ]:
from sklearn.metrics import classification_report
y_pred= clf.predict(X_test)
print(classification_report(y_true=y_test,y_pred=y_pred))

In [ ]:
X_train.columns

In [ ]:
entrada = np.array([[12, 3, 0,1,0,1,0,0,0]])

# Realizar la clasificación
prediccion = clf.predict(entrada)
prediccion

In [ ]:
X_train.Tamaño_OE.value_counts()

#Veamos que pasa con un poco de feature engineering

In [ ]:
# Split en train y test
X= df_final[['Peso', 'Tamaño_OE']]
y= df_final['Sabor_pescado']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
# Entrenar el arbol
dt = tree.DecisionTreeClassifier(min_samples_leaf=50,max_depth=6)
dt.fit(X_train,y_train)

In [ ]:
entrada = np.array([[30, 2]])

# Realizar la clasificación
prediccion = dt.predict(entrada)
prediccion

In [ ]:
import cv2 #open-cv2
tree.export_graphviz(
dt,
out_file="ejemplo.dot",
feature_names=["Peso","Tamaño_OE"],
rounded=True,
filled=True
)

#Convierto el dot a png
! dot -Tpng ejemplo.dot -o ejemplo.png

#Ploteamos el png
img = cv2.imread('ejemplo.png')
plt.figure(figsize = (20, 10))
plt.imshow(img)

In [ ]:
y_train.value_counts()

In [ ]:

# En un árbol de decisión, puedes obtener la importancia de las variables utilizando el atributo feature_importances_ del clasificador entrenado.
# nos devuelve la importancia la de las variables en un árbol de decisión y se basa en la medida de impureza utilizada durante la construcción del árbol, en este caso gini.

importances = dt.feature_importances_

# Obtener los nombres de las variables
feature_names = X_train.columns

# Crear un DataFrame con las importancias de las variables
importance_df = pd.DataFrame({'Variable': feature_names, 'Importance': importances})

# Ordenar el DataFrame por importancia descendente
importance_df = importance_df.sort_values('Importance', ascending=False)


print(importance_df)

In [ ]:
from sklearn.metrics import classification_report
y_pred= dt.predict(X_test)
print(classification_report(y_true=y_test,y_pred=y_pred))

##Generalmente, si tuviesemos algo tan fácil no vale la pena hacer un modelo de ML. Con un if-else bastaría para tener resultados similares.

##Guardar un modelo para usarlo a futuro.

In [ ]:
import pickle

# Supongamos que tienes un modelo entrenado llamado "dt"
modelo = dt

# Guardamos el modelo en un archivo utilizando pickle
with open('modelo_dt.pkl', 'wb') as archivo:
    pickle.dump(modelo, archivo)

In [ ]:
# ahora veamos como usarlo a futuro

# Cargamos el modelo
with open('modelo_dt.pkl', 'rb') as archivo:
    modelo_cargado = pickle.load(archivo)

# Utilizar el modelo cargado para hacer predicciones, por ejemplo:
predicciones = modelo_cargado.predict(X_test)
predicciones

In [ ]:
modelo_cargado.predict(np.array([[80,3]]))
